I already ingested data of the orders using the databricks data ingestion through amazon s3 bucket 

In [0]:
display(dbutils.fs.ls('s3://de-prj'))

path,name,size,modificationTime
s3://de-prj/csv_files/,csv_files/,0,1792141823382
s3://de-prj/json_files/,json_files/,0,1792141823382


In [18]:
%sql
use catalog aws ;
use schema test ;

OK

In [0]:
def create_table(path, table_name):
    df = spark.read.csv(
        path,
        header=True,
        inferSchema=True
    )
    df.createOrReplaceTempView(table_name)
for file in dbutils.fs.ls('s3://de-prj/csv_files'):
    if not file.name.endswith('.csv'):
        continue
    table_name = file.name.replace('.csv', '')
    create_table(
        file.path,
        table_name
    )
    print(f"Created {table_name}")

Created dim_customers
Created dim_delivery_agents
Created dim_locations
Created dim_menu_items
Created dim_restaurants


In [24]:
%sql
show tables

database,tableName,isTemporary
,dim_customers,True
,dim_delivery_agents,True
,dim_locations,True
,dim_menu_items,True
,dim_restaurants,True


In [0]:
%sql
ALTER TABLE aws.project.bronze_orders SET TBLPROPERTIES (delta.enableRowTracking = true); -- Bcoz it is reading from the managed table which is created by the automatic data ingestion from s3 using the databricks data ingestion to create the bronze table

OK

In [2]:
%sql
select * from aws.project.silver_orders
where agent_id is null ;

agent_id,customer_id,delivery_location_id,items_ordered,order_id,order_status,payment_method,restaurant_id,timestamp,tip,total_amount
